In [2]:
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate
!pip install -U bitsandbytes>=0.46.1

In [3]:
import json
import random
import pandas as pd
from datetime import datetime, timedelta

def generate_digital_phenotype():
    data = {
        "user_id": "medchron_user_01",
        "device_sync_time": str(datetime.now()),
        "activity_logs": []
    }
    
    start_time = datetime(2026, 2, 20, 0, 0) # Starting at midnight
    
    for minute in range(1440): # 24 hours of data
        timestamp = (start_time + timedelta(minutes=minute))
        hour = timestamp.hour
        
        # 1. Simulate Phone Inactivity (Sleep Proxy)
        # Normal sleep: 00:00 - 07:00. If usage occurs, it's a "Sleep Interruption"
        is_active = False
        if hour >= 7 and hour < 23:
            is_active = random.random() > 0.7 # 30% chance of being on phone during day
        elif hour >= 1 and hour < 4:
            is_active = random.random() > 0.95 # 5% chance of late-night "doomscrolling"

        # 2. App Switching Density (Cognitive Load Proxy)
        # Higher switches per minute = Higher potential stress
        app_switches = random.randint(5, 15) if (is_active and 9 <= hour <= 17) else random.randint(0, 3)

        # 3. Laptop Processing Power (Workload Proxy)
        cpu_load = random.randint(40, 90) if (20 <= hour <= 23) else random.randint(5, 20)

        data["activity_logs"].append({
            "timestamp": timestamp.strftime("%H:%M"),
            "phone_active": is_active,
            "app_switches": app_switches,
            "laptop_cpu_percent": cpu_load,
            "ambient_light_lux": 300 if (7 <= hour <= 18) else 10 # Low light = Night usage
        })

    with open('digital_phenotype_logs.json', 'w') as f:
        json.dump(data, f, indent=4)
    print("✅ Success! digital_phenotype_logs.json created in /kaggle/working/")

generate_digital_phenotype()

✅ Success! digital_phenotype_logs.json created in /kaggle/working/


In [4]:
import os
import torch
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Authenticate (Ensure HF_TOKEN is enabled in Add-ons -> Secrets)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# 2. Precise Model ID from Hugging Face
model_id = "google/medgemma-1.5-4b-it"

# 3. 4-bit Config (Optimized for your Dual T4 GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# 4. Load Model & Tokenizer
print(f" Initializing download for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print(" MedGemma 1.5 is now loaded in 4-bit VRAM!")

# 5. MedChron Analysis Function
def run_medchron_analysis(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extracting patterns for the AI to reason over
    logs = data['activity_logs']
    # Filtering for phone activity between 1 AM and 5 AM
    late_night = [l['timestamp'] for l in logs if l['phone_active'] and "01:00" <= l['timestamp'] <= "05:00"]
    # Filtering for high app switching
    high_switching = [l['timestamp'] for l in logs if l['app_switches'] > 12]

    # Creating a medical-style prompt
    prompt = f"""<start_of_turn>user
    You are a specialist in Digital Phenotyping and Behavioral Health. 
    Analyze the following device usage metadata for a single 24-hour period:
    
    - Observed sleep fragmentation (Phone active during rest hours): {late_night[:8]}
    - High cognitive load spikes (App switching >12/min): {high_switching[:8]}
    - Sustained laptop CPU usage (>80%) detected during late-evening hours.

    Based on these digital biomarkers, provide a professional 'MedChron Live Report'. 
    Discuss the user's circadian rhythm disruption, potential cognitive fatigue, and burnout risk.<end_of_turn>
    <start_of_turn>model"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate response
    outputs = model.generate(
        **inputs, 
        max_new_tokens=400, 
        temperature=0.7, 
        do_sample=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 6. Execute if data is ready
# Check if the file exists in the current directory
file_path = 'digital_phenotype_logs.json'
if os.path.exists(file_path):
    print("\n---  Generating MedChron Analysis ---")
    report = run_medchron_analysis(file_path)
    print("\n" + "="*50)
    print(report)
    print("="*50)
else:
    print(f" Error: {file_path} not found in {os.getcwd()}")

 Initializing download for google/medgemma-1.5-4b-it...


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


 MedGemma 1.5 is now loaded in 4-bit VRAM!

---  Generating MedChron Analysis ---

user
    You are a specialist in Digital Phenotyping and Behavioral Health. 
    Analyze the following device usage metadata for a single 24-hour period:
    
    - Observed sleep fragmentation (Phone active during rest hours): ['01:08', '01:22', '01:26', '01:28', '01:37', '01:51', '02:33', '02:44']
    - High cognitive load spikes (App switching >12/min): ['09:13', '09:25', '10:09', '10:15', '10:25', '10:33', '10:52', '11:16']
    - Sustained laptop CPU usage (>80%) detected during late-evening hours.

    Based on these digital biomarkers, provide a professional 'MedChron Live Report'. 
    Discuss the user's circadian rhythm disruption, potential cognitive fatigue, and burnout risk.
    model
    <unused94>thought
Here's a thinking process that could lead to the MedChron Live Report:

1.  **Understand the Goal:** The request asks for a professional "MedChron Live Report" based on specific digital biom

In [5]:
# 1. Install Streamlit and a tool to tunnel it so you can see it
!pip install -q streamlit pyngrok

# 2. Create the Dashboard Script
with open('app.py', 'w') as f:
    f.write("""
import streamlit as st
import json
import pandas as pd

st.set_page_config(page_title="MedChron Live", page_icon="🧠")

st.title("MedChron: Mental Health Dashboard")
st.subheader("Passive Behavioral Phenotyping via MedGemma 1.5")

# --- Sidebar: User Info ---
st.sidebar.header("User Profile")
st.sidebar.text("ID: nihal_student_unf")
st.sidebar.progress(75, text="Cognitive Load: High")

# --- Main Metrics ---
col1, col2, col3 = st.columns(3)
col1.metric("Sleep Fragmentation", "High", "+20%")
col2.metric("App Switching", "14/min", "Critical")
col3.metric("Brain Rest", "4.2 hrs", "-1.5 hrs")

# --- The AI Live Report ---
st.header("Live Mental Health Narrative")
# In a real app, this text comes from your MedGemma output
st.info(\"\"\"
**AI Observation:** Based on 24-hour device logs, you are showing signs of 
circadian rhythm disruption. Your phone activity between 01:00 and 03:00 
correlated with high laptop CPU usage suggests a 'burnout' state. 
**Recommendation:** Schedule a 20-minute 'Digital Reset' and aim for a 
consistent device-off time tonight.
\"\"\")

# --- Activity Visualization ---
st.header("Digital Activity Trends")
# Simulating some data
chart_data = pd.DataFrame({
    'Hour': list(range(24)),
    'Stress Level': [10, 5, 5, 2, 2, 5, 10, 20, 40, 60, 80, 75, 70, 65, 85, 90, 95, 80, 60, 50, 65, 80, 90, 95]
})
st.line_chart(chart_data.set_index('Hour'))

st.success("Data is being processed locally on Edge AI to ensure total privacy.")
""")

print("app.py created. You are ready to build the UI!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 84.8 MB/s eta 0:00:00:00:0100:01
app.py created. You are ready to build the UI!


In [4]:
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate

In [6]:
import os
import torch
import json
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --- PHASE 1: AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# --- PHASE 2: DATA GENERATION (The Digital Phenotype) ---
def generate_fresh_logs():
    print("📊 Phase 1: Generating raw device activity logs...")
    data = {"user_id": "nihal_unf_student", "activity_logs": []}
    # Simulating a high-stress day with 4 hours of late-night laptop work
    for h in range(24):
        is_active = True if (1 <= h <= 3) or (9 <= h <= 23) else False
        switches = 15 if (10 <= h <= 15) else 3
        cpu = 90 if (21 <= h <= 23) else 20
        data["activity_logs"].append({
            "timestamp": f"{h:02d}:00",
            "phone_active": is_active,
            "app_switches": switches,
            "laptop_cpu": cpu
        })
    with open('full_loop_data.json', 'w') as f:
        json.dump(data, f)
    return 'full_loop_data.json'

log_file = generate_fresh_logs()

# --- PHASE 3: LOAD MEDGEMMA 1.5 ---
model_id = "google/medgemma-1.5-4b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"🧠 Phase 2: Loading MedGemma 1.5 for reasoning...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)

# --- PHASE 4: THE REASONING LOOP ---
def run_inference(file_path):
    with open(file_path, 'r') as f:
        logs = json.load(f)['activity_logs']
    
    # Extract markers for the prompt
    night_usage = [l['timestamp'] for l in logs if l['phone_active'] and "01:00" <= l['timestamp'] <= "04:00"]
    
    prompt = f"""<start_of_turn>user
    Identify mental health risks from these digital biomarkers:
    - User was on phone during typical sleep hours at: {night_usage}
    - Laptop CPU was at 90% capacity from 10 PM to Midnight.
    - App switching frequency is 3x higher than baseline during morning hours.
    
    Provide a professional MedChron Live Report.<end_of_turn>
    <start_of_turn>model"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=350, temperature=0.7)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("📝 Phase 3: Analyzing patterns...")
final_report = run_inference(log_file)

# --- PHASE 5: SAVE FOR UI ---
print("🖥️ Phase 4: Finalizing UI Snapshot...")
ui_output = {
    "report_text": final_report,
    "status": "Burnout Risk: High",
    "last_updated": str(datetime.now())
}
with open('ui_snapshot.json', 'w') as f:
    json.dump(ui_output, f, indent=4)

print("\n" + "="*30)
print("🚀 FULL LOOP COMPLETE")
print(final_report)
print("="*30)

📊 Phase 1: Generating raw device activity logs...
🧠 Phase 2: Loading MedGemma 1.5 for reasoning...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


📝 Phase 3: Analyzing patterns...
🖥️ Phase 4: Finalizing UI Snapshot...

🚀 FULL LOOP COMPLETE
user
    Identify mental health risks from these digital biomarkers:
    - User was on phone during typical sleep hours at: ['01:00', '02:00', '03:00']
    - Laptop CPU was at 90% capacity from 10 PM to Midnight.
    - App switching frequency is 3x higher than baseline during morning hours.
    
    Provide a professional MedChron Live Report.
    model
<unused94>thought
Here's a thinking process that could lead to the MedChron Live Report:

1.  **Understand the Goal:** The request asks for a professional MedChron Live Report identifying mental health risks based on three specific digital biomarker data points. The report needs to be structured, professional, and clearly link the data to potential mental health concerns.

2.  **Identify the Target Audience and Purpose:** The report is likely for a healthcare provider, patient, or possibly an internal team. The purpose is to raise awareness of p